In [1]:
!pip install pandas
!pip install transformers==4.38.2
!pip install tensorboard
!pip install --ignore-installed blinker flask 
!pip install flask-cors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 352.7 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 120.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 792.7/792.7 kB 148.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 236.5 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 388.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 488.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 464.9 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip
  Using c

In [2]:
import os
os.chdir('llm-localizer')

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
from IPython.display import display, HTML

def visualize_vulnerabilities_blue(code_lines, probabilities, labels):
    html_lines = []

    for line, prob, label in zip(code_lines, probabilities, labels):
        prob = max(0.0, min(1.0, prob))
        blue_intensity = int(255 * prob)

        # Use a white base with blue overlay
        background_color = f"rgba(0, 0, {blue_intensity}, 0.4)"
        border = "1px solid red" if round(label) == 1 else "none"

        safe_line = (
            line.replace("&", "&amp;")
                .replace("<", "&lt;")
                .replace(">", "&gt;")
        )

        html_lines.append(
            f"""<div style="
                background: white;
                background-color: {background_color};
                border: {border};
                font-family: monospace;
                white-space: pre;
                line-height: 1.2;
            ">{safe_line}</div>"""
        )

    html_output = "<div style='line-height: 1.2'>" + "\n".join(html_lines) + "</div>"
    display(HTML(html_output))


In [6]:
import os
import shutil

import torch
import pandas as pd
from torch.utils.data import DataLoader

from components.llm import LLM
from components.llm_utils import LLMModels, LLMInfo
from components.localization_model import LocalizationTransformer, Config, LastHiddenStatesDataset, \
    TrainValidationSplit, ConfigFactory
from components.prompt import Driver as Tokenizer


class LocalizeVulnerabilities:
    def __init__(self,
                 dataset_name: str,
                 dataset_version: str,
                 checkpoint_config: Config,
                 pre_code_part: str = None,
                 post_code_part: str = None,
                 mode: str = "inference"
                 ):
        self.dataset_name = dataset_name
        self.dataset_version = dataset_version
        self.checkpoint_config = checkpoint_config
        self.pre_code_part = pre_code_part
        self.post_code_part = post_code_part
        self.mode = mode

        self.device = "cuda:0"

    def create_new_dataset(self, code):
        def delete_folder(folder_path):
            if os.path.exists(folder_path) and os.path.isdir(folder_path):
                shutil.rmtree(folder_path)
                print(f"Deleted folder: {folder_path}")
            else:
                raise FileNotFoundError(f"Folder not found: {folder_path}")
        delete_folder(f"{os.getcwd()}/{self.checkpoint_config.tensor_path}/{self.dataset_name}/{self.dataset_version}")
        df_path = f"{os.getcwd()}/{self.checkpoint_config.dataset_path}/{self.dataset_name}/{self.dataset_version}/{self.dataset_name}.csv"
        
        data = [{'item_index': 0, 'source_code': code, 'vuln_lines': "[]"}]
        df = pd.DataFrame(data)
        df.to_csv(df_path)
        return df

    def tokenize(self):
        Tokenizer(tensor_path=self.checkpoint_config.tensor_path,
                  dataset_path=self.checkpoint_config.dataset_path,
                  dataset_version=self.dataset_version,
                  dataset_name=self.dataset_name,
                  llm_model=self.checkpoint_config.llm_model,
                  pre_code_part=self.pre_code_part,
                  post_code_part=self.post_code_part,
                  standardize_df=False)

    def llm_inference(self):
        llm = LLM(tensor_path=self.checkpoint_config.tensor_path,
                  dataset_version=self.dataset_version,
                  dataset_name=self.dataset_name,
                  tokens_type=self.checkpoint_config.tokens_type,
                  llm_model=self.checkpoint_config.llm_model,
                  device=self.device) 
        llm.get_and_save_last_hidden_states()

    def localization(self):
        model = LocalizationTransformer(
            self.checkpoint_config.num_layers_projection,
            self.checkpoint_config.num_layers_encoder,
            self.checkpoint_config.num_layers_dim_reduce,
            LLMInfo(self.checkpoint_config.llm_model).get_hidden_size(),
            self.checkpoint_config.num_head,
            self.checkpoint_config.target_dim,
            self.checkpoint_config.dim_reduce_type,
            self.checkpoint_config.seed,
            self.device

        ).to(self.device)

        cwd = os.getcwd()

        base_model_checkpoint_dir = f"{cwd}/{self.checkpoint_config.outputs_path}/{self.checkpoint_config.dataset_name}/{self.checkpoint_config.dataset_version}/{LLMModels.get_model_nickname(self.checkpoint_config.llm_model)}/{self.checkpoint_config.exp_config}/checkpoints/fold_{self.checkpoint_config.fold_index}"
        if not os.path.isdir(base_model_checkpoint_dir):
            raise FileNotFoundError("Checkpoint folder does not exist")
        base_checkpoint_files = [f for f in os.listdir(base_model_checkpoint_dir) if
                                 os.path.isfile(os.path.join(base_model_checkpoint_dir, f))]
        if len(base_checkpoint_files) > 0:
            latest_epoch = 0
            for file in base_checkpoint_files:
                epoch_of_file = int(file.split('.')[0])
                if epoch_of_file > latest_epoch:
                    latest_epoch = epoch_of_file
            checkpoint = torch.load(f'{base_model_checkpoint_dir}/{latest_epoch}.pt', map_location=self.device)
            model.load_state_dict(checkpoint['model_state_dict'])
        else:
            raise FileNotFoundError("No checkpoints found for the given model configuration")

        # Create the TrainValidationSplit object
        tvs = TrainValidationSplit(self.checkpoint_config.tensor_path,
                                   self.dataset_version,
                                   self.dataset_name,
                                   self.checkpoint_config.tokens_type,
                                   self.checkpoint_config.llm_model,
                                   1,
                                   self.checkpoint_config.seed)

        # Retrieve the train/validation indices for your chosen fold
        train_validation_indices = tvs.get_train_validation_indices_for_fold(0)

        inference_dataset = LastHiddenStatesDataset(
            self.checkpoint_config.tensor_path,
            self.checkpoint_config.dataset_path,
            self.dataset_version,
            self.dataset_name,
            self.checkpoint_config.tokens_type,
            self.checkpoint_config.llm_model,
            train_validation_indices,
            dataset_type="validation"
        )

        inference_loader = DataLoader(inference_dataset, batch_size=1, shuffle=False)

        results = []
        model.eval()
        with torch.no_grad():
            for step, (last_hidden_state, code_tokens_length, line_split_lengths, line_labels) in enumerate(
                    inference_loader):
                last_hidden_state = last_hidden_state.to(self.device)
                code_tokens_length = code_tokens_length.to(self.device)
                line_split_lengths = line_split_lengths.to(self.device)
                line_labels = line_labels.to(self.device)

                outputs = model(last_hidden_state, code_tokens_length, line_split_lengths)

                mask = (line_labels != -1)
                valid_outputs = outputs[mask]

                preds_prob = torch.sigmoid(valid_outputs)
                preds = (preds_prob >= 0.5).long()

                if self.mode == "inference":
                    results.append({'Output': valid_outputs.tolist(), 'Probabilities': preds_prob.tolist(), 'Classification': preds.tolist()})
                elif self.mode == "evaluation":
                    valid_labels = line_labels[mask].float()
                    results.append({'Output': valid_outputs.tolist(), 'Probabilities': preds_prob.tolist(), 'Classification': preds.tolist(), 'Actual': valid_labels.tolist()})
                else:
                    raise Exception("There is no such mode available!")
        return results


if __name__ == '__main__':
    base_model_fold = 8  # replace the 0 with the fold you want to load
    base_model_config = ConfigFactory(exp_config='exp10',
                                      dataset_version='v2',
                                      dataset_name='solidity',
                                      llm_models_list=[LLMModels.DEEPSEEK_R1_DISTILL_QWEN_14B],
                                      layer_conf=2,
                                      target_dim_list=[1024],
                                      dim_reduce_type='gru',
                                      max_learning_rate_list=[1e-4],
                                      criterion="BCEWithLogitsLoss").get_generated_configs()[base_model_fold]

    localizer = LocalizeVulnerabilities(
        dataset_name="localize_infer",
        dataset_version="v1",
        checkpoint_config=base_model_config,
        pre_code_part="Smart contracts written in Solidity language may contain vulnerabilities such as DelegateCall, Arithmetic/Integer Overflow and Underflow, Nested Call, Reentrancy, Timestamp Dependency, TxOrigin, Transaction Order Dependency, Unchecked Call, Unprotected Suicide, Frozen Ether, Bad Randomness, Denial of service, Front Running, Short Address and other vulnerabilities. Analyze the following solidity smart contract for security vulnerabilities, bugs, and faulty logic. Identify all problematic lines and explain the risks associated with each.",
        post_code_part="",
        mode="evaluation"
    )

    new_code = r"""pragma solidity ^0.6.0;
/**
 * @title Core
 * @dev Solidity version 0.5.x prevents to mark as view
 * @dev functions using delegate call.
 *
 * @author Cyril Lapinte - <cyril.lapinte@openfiz.com>
 *
 * Error messages
 *   CO01: Only Proxy may access the function
 *   CO02: Address 0 is an invalid delegate address
 *   CO03: Delegatecall should be successful
 *   CO04: DelegateId must be greater than 0
 *   CO05: Proxy must exist
 *   CO06: Proxy must be already defined
 *   CO07: Proxy update must be successful
 **/
contract Core is Storage {
  using BytesConvert for bytes;
  modifier onlyProxy {
    require(delegates[proxyDelegateIds[msg.sender]] != address(0), ""CO01"");
    _;
  }
  function delegateCall(address _proxy) internal returns (bool status)
  {
    uint256 delegateId = proxyDelegateIds[_proxy];
    address delegate = delegates[delegateId];
    require(delegate != address(0), ""CO02"");
    // solhint-disable-next-line avoid-low-level-calls
    (status, ) = delegate.delegatecall(msg.data);
    require(status, ""CO03"");
  }
  function delegateCallUint256(address _proxy)
    internal returns (uint256)
  {
    return delegateCallBytes(_proxy).toUint256();
  }
  function delegateCallBytes(address _proxy)
    internal returns (bytes memory result)
  {
    bool status;
    uint256 delegateId = proxyDelegateIds[_proxy];
    address delegate = delegates[delegateId];
    require(delegate != address(0), ""CO02"");
    // solhint-disable-next-line avoid-low-level-calls
    (status, result) = delegate.delegatecall(msg.data);
    require(status, ""CO03"");
  }
  function defineDelegateInternal(uint256 _delegateId, address _delegate) internal returns (bool) {
    require(_delegateId != 0, ""CO04"");
    delegates[_delegateId] = _delegate;
    return true;
  }
  function defineProxyInternal(address _proxy, uint256 _delegateId)
    virtual internal returns (bool)
  {
    require(delegates[_delegateId] != address(0), ""CO02"");
    require(_proxy != address(0), ""CO05"");
    proxyDelegateIds[_proxy] = _delegateId;
    return true;
  }
  function migrateProxyInternal(address _proxy, address _newCore)
    internal returns (bool)
  {
    require(proxyDelegateIds[_proxy] != 0, ""CO06"");
    require(Proxy(_proxy).updateCore(_newCore), ""CO07"");
    return true;
  }
  function removeProxyInternal(address _proxy)
    internal returns (bool)
  {
    require(proxyDelegateIds[_proxy] != 0, ""CO06"");
    delete proxyDelegateIds[_proxy];
    return true;
  }
}"""
    new_df = localizer.create_new_dataset(new_code)
    print(new_df.head())
    localizer.tokenize()
    localizer.llm_inference()
    results = localizer.localization()

Deleted folder: /workspace/llm-localizer/data/tensors/localize_infer/v1
   item_index                                        source_code vuln_lines
0           0  pragma solidity ^0.6.0;\n/**\n * @title Core\n...         []


Creating tensors for LLM: 100%|██████████| 1/1 [00:00<00:00,  2.21it/s]


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Inferencing: 100%|██████████| 1/1 [00:00<00:00,  1.71it/s]


In [7]:
import pandas as pd

dataset_path = f"{os.getcwd()}/data/dataset/localize_infer/v1/localize_infer.csv"
df = pd.read_csv(dataset_path)

for i in range(len(results)):
    result = results[i]
    print(f"Code {i + 1}") 
    code = df.loc[df['item_index'] == i, 'source_code'].iloc[0]
    probabilities = result['Probabilities']
    if 'Actual' in result:
        actual = result['Actual']
    else:
        actual = [0 for _ in range(len(probabilities))]
    visualize_vulnerabilities_blue(code.splitlines(), probabilities, actual)

Code 1
